In [1]:
import json

def load(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(x) for x in f]

data = load("/Users/JL/Desktop/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_multi_turn_base.json")



In [2]:
data

[{'id': 'multi_turn_base_0',
  'question': [[{'role': 'user',
     'content': "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"}],
   [{'role': 'user',
     'content': "Perform a detailed search using grep to identify sections in the file pertaining to 'budget analysis'."}],
   [{'role': 'user',
     'content': "Upon identifying the requisite 'budget analysis' content, sort the 'final_report.pdf' by line for improved clarity and comprehension."}],
   [{'role': 'user',
     'content': "Move 'previous_report.pdf' in document directory to temp as well and having final report also there, proceed to juxtapose it with 'previous_report.pdf' to detect any critical alterations."}]],
  'initial_config': {'GorillaFileSystem': {'root': {'workspace': {'type': 'directory',
      'contents': {'document': {'type': 'directory',
        'contents': {'final_report.pdf': {'type': 'file',
          'content': 'Year2024 This is the final r

# BFCL v3 Multi-Turn Base 数据探索

以下分析依托 `gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_multi_turn_base.json` 数据集，整理其结构与核心统计信息，便于后续模型或评测研究。


In [3]:
from collections import Counter
import pandas as pd

records = data
num_records = len(records)
print(f"Loaded {num_records} multi-turn tasks.")
print(f"Top-level keys: {sorted(records[0].keys())}")


Loaded 200 multi-turn tasks.
Top-level keys: ['id', 'initial_config', 'involved_classes', 'path', 'question']


In [4]:
records[0]['question']

[[{'role': 'user',
   'content': "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"}],
 [{'role': 'user',
   'content': "Perform a detailed search using grep to identify sections in the file pertaining to 'budget analysis'."}],
 [{'role': 'user',
   'content': "Upon identifying the requisite 'budget analysis' content, sort the 'final_report.pdf' by line for improved clarity and comprehension."}],
 [{'role': 'user',
   'content': "Move 'previous_report.pdf' in document directory to temp as well and having final report also there, proceed to juxtapose it with 'previous_report.pdf' to detect any critical alterations."}]]

In [5]:
records_df = pd.DataFrame({
    'id': [item['id'] for item in records],
    'turn_groups': [len(item['question']) for item in records],
    'messages': [sum(len(turn) for turn in item['question']) for item in records],
    'path_length': [len(item['path']) for item in records],
    'involved_classes': [item['involved_classes'] for item in records],
    'path': [item['path'] for item in records],
})
records_df


,id,turn_groups,messages,path_length,involved_classes,path
0,multi_turn_base_0,4,4,6,"[TwitterAPI, GorillaFileSystem]","[GorillaFileSystem.find, GorillaFileSystem.mv,..."
1,multi_turn_base_1,4,4,5,[GorillaFileSystem],"[GorillaFileSystem.ls, GorillaFileSystem.find,..."
2,multi_turn_base_2,5,5,7,"[TicketAPI, GorillaFileSystem]","[GorillaFileSystem.touch, GorillaFileSystem.ec..."
3,multi_turn_base_3,2,2,3,[GorillaFileSystem],"[GorillaFileSystem.cd, GorillaFileSystem.find,..."
4,multi_turn_base_4,3,3,5,"[TwitterAPI, GorillaFileSystem]","[GorillaFileSystem.cd, GorillaFileSystem.echo,..."
...,...,...,...,...,...,...
195,multi_turn_base_195,4,4,5,"[TwitterAPI, TravelAPI]","[TravelAPI.get_nearest_airport_by_city, Travel..."
196,multi_turn_base_196,5,5,7,"[TicketAPI, TravelAPI]","[TravelAPI.compute_exchange_rate, TravelAPI.se..."
197,multi_turn_base_197,3,3,7,"[MessageAPI, TravelAPI]","[TravelAPI.register_credit_card, TravelAPI.pur..."
198,multi_turn_base_198,1,1,4,"[TwitterAPI, TravelAPI]","[TravelAPI.get_flight_cost, TravelAPI.book_fli..."


In [6]:
records_df.iloc[0]

id                                                  multi_turn_base_0
turn_groups                                                         4
messages                                                            4
path_length                                                         6
involved_classes                      [TwitterAPI, GorillaFileSystem]
path                [GorillaFileSystem.find, GorillaFileSystem.mv,...
Name: 0, dtype: object

In [7]:
records_df.iloc[0]['path']

['GorillaFileSystem.find',
 'GorillaFileSystem.mv',
 'GorillaFileSystem.grep',
 'GorillaFileSystem.sort',
 'GorillaFileSystem.diff',
 'TwitterAPI.post_tweet']

In [9]:
# 获取所有path中的唯一元素（即所有函数调用名称），可以使用set合并后再去重
from collections import defaultdict

unique_funcs = set()
for path in records_df['path']:
    unique_funcs.update(path)
unique_funcs = sorted(unique_funcs)

# 按 A.B 中的 A 归组
grouped_funcs = defaultdict(list)
for func in unique_funcs:
    if '.' in func:
        group, method = func.split('.', 1)
        grouped_funcs[group].append(func)
    else:
        grouped_funcs['(no group)'].append(func)

# 输出分组结果
for group, funcs in grouped_funcs.items():
    print(f"{group}:")
    for func in funcs:
        print(f"  {func}")


GorillaFileSystem:
  GorillaFileSystem.cat
  GorillaFileSystem.cd
  GorillaFileSystem.cp
  GorillaFileSystem.diff
  GorillaFileSystem.echo
  GorillaFileSystem.find
  GorillaFileSystem.grep
  GorillaFileSystem.ls
  GorillaFileSystem.mkdir
  GorillaFileSystem.mv
  GorillaFileSystem.rm
  GorillaFileSystem.sort
  GorillaFileSystem.tail
  GorillaFileSystem.touch
  GorillaFileSystem.wc
MathAPI:
  MathAPI.logarithm
  MathAPI.mean
  MathAPI.standard_deviation
MessageAPI:
  MessageAPI.delete_message
  MessageAPI.send_message
  MessageAPI.view_messages_received
TicketAPI:
  TicketAPI.close_ticket
  TicketAPI.create_ticket
  TicketAPI.edit_ticket
  TicketAPI.get_ticket
  TicketAPI.resolve_ticket
TradingBot:
  TradingBot.add_stock_to_watchlist
  TradingBot.cancel_order
  TradingBot.fund_account
  TradingBot.get_account_info
  TradingBot.get_available_stocks
  TradingBot.get_current_time
  TradingBot.get_order_details
  TradingBot.get_stock_info
  TradingBot.get_symbol_by_name
  TradingBot.get_watc

In [10]:
for i, item in enumerate(records[:20]):
    print(f"Record {i}:")
    print(item["question"])
    print("="*60)


Record 0:
[[{'role': 'user', 'content': "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"}], [{'role': 'user', 'content': "Perform a detailed search using grep to identify sections in the file pertaining to 'budget analysis'."}], [{'role': 'user', 'content': "Upon identifying the requisite 'budget analysis' content, sort the 'final_report.pdf' by line for improved clarity and comprehension."}], [{'role': 'user', 'content': "Move 'previous_report.pdf' in document directory to temp as well and having final report also there, proceed to juxtapose it with 'previous_report.pdf' to detect any critical alterations."}]]
Record 1:
[[{'role': 'user', 'content': 'I am alex. Check if the current directory is under my name and list all the visible and hidden contents in the current directory now, please.'}], [{'role': 'user', 'content': "Go to workspace directory and move one of the 'log.txt' files into a new directory 'archive'."

# BFCL v3 Multi-Turn Missing Parameters 数据探索

以下分析依托 `gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_multi_turn_miss_param.json` 数据集，整理其结构与核心统计信息，便于后续模型或评测研究。


In [ ]:
data_miss_param = load("/Users/JL/Desktop/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/bfcl_eval/data/BFCL_v3_multi_turn_miss_param.json")
data_miss_param




In [ ]:
from collections import Counter
import pandas as pd

records = data_miss_param
num_records = len(records)
print(f"Loaded {num_records} multi-turn tasks.")
print(f"Top-level keys: {sorted(records[0].keys())}")


In [ ]:
records_df = pd.DataFrame(records)
records_df


In [ ]:
records_df.iloc[0]